# K-Means Clustering
**Dataset:** Credit Card Transactions (1.3M rows → 781 customers after aggregation)

K-Means groups customers into K clusters. Customers inside each group are similar to each other. Customers across groups are different from each other.

---
**Before running this notebook, make sure you have:**
- `credit_card_transactions.csv` in the same folder
- Run `pip install scikit-learn` if needed

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

plt.style.use('ggplot')

## 2. Load and prepare the data
We load the raw file, clean it, and collapse 1.3M rows into one row per customer.

In [ ]:
# Load
df = pd.read_csv('credit_card_transactions.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Parse dates
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob']                   = pd.to_datetime(df['dob'], errors='coerce')
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days / 365.25

print('Raw data loaded:', df.shape)

In [ ]:
# Remove outliers using IQR method (from our EDA findings)
for col in ['amt', 'age', 'city_pop']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    df     = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]

print('After outlier removal:', df.shape)
print('Unique customers:', df['cc_num'].nunique())

In [ ]:
# Aggregate: one row per customer
# This is the key step — collapse 1.3M transactions into 781 customer rows
ref_date = df['trans_date_trans_time'].max() + pd.Timedelta(days=1)

rfm = df.groupby('cc_num').agg(
    Recency   = ('trans_date_trans_time', lambda x: (ref_date - x.max()).days),
    Frequency = ('trans_num', 'count'),
    Monetary  = ('amt', 'sum'),
).reset_index()

print('Customer table shape:', rfm.shape)
print()
print(rfm.head())

In [ ]:
# Fix skewness — Recency skewness is 5.15, log1p brings it down
rfm['Recency_log']  = np.log1p(rfm['Recency'])
rfm['Monetary_log'] = np.log1p(rfm['Monetary'])

print('Skewness before and after:')
print(f'  Recency  : {rfm["Recency"].skew():.2f}  →  {rfm["Recency_log"].skew():.2f}')
print(f'  Monetary : {rfm["Monetary"].skew():.2f}  →  {rfm["Monetary_log"].skew():.2f}')

In [ ]:
# Scale with RobustScaler
# K-Means uses distances — unscaled columns with big numbers dominate
cols_to_cluster = ['Recency_log', 'Frequency', 'Monetary_log']

scaler = RobustScaler()
X = scaler.fit_transform(rfm[cols_to_cluster])

print('Scaled array shape:', X.shape)
print('Each column now has similar range — no single column dominates')

## 3. Find the best K

### Method A — Elbow Method (WCSS)
WCSS = Within-Cluster Sum of Squares. Lower = tighter clusters.
Plot it and look for the bend (elbow).

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    km.fit(X)
    wcss.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, marker='o', color='#2E75B6', linewidth=2, markersize=8)
plt.title('Elbow Method — look for the bend')
plt.xlabel('K (number of clusters)')
plt.ylabel('WCSS (lower = tighter)')
plt.xticks(K_range)
plt.tight_layout()
plt.show()

print('WCSS values:')
for k, w in zip(K_range, wcss):
    print(f'  K={k}  WCSS={w:.1f}')

### Method B — Silhouette Score
Ranges from -1 to +1. Higher = customers fit their own cluster well.
Pick the K with the highest score.

In [ ]:
sil_scores = []
K_range2   = range(2, 11)

for k in K_range2:
    km     = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

plt.figure(figsize=(9, 5))
plt.plot(K_range2, sil_scores, marker='s', color='#1D9E75', linewidth=2, markersize=8)
plt.title('Silhouette Score — higher = better')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.xticks(K_range2)
plt.tight_layout()
plt.show()

print('Silhouette scores:')
for k, s in zip(K_range2, sil_scores):
    print(f'  K={k}  Score={s:.3f}')

best_k = list(K_range2)[sil_scores.index(max(sil_scores))]
print(f'
Highest silhouette at K={best_k}')
print('We will use K=4 for 4 meaningful business segments')

## 4. Train the final K-Means model

In [ ]:
K = 4  # change this if your charts suggest a different number

km_final = KMeans(n_clusters=K, init='k-means++', random_state=42, n_init=10)
rfm['Cluster'] = km_final.fit_predict(X)

print(f'Model trained with K={K}')
print()
print('Customers per cluster:')
print(rfm['Cluster'].value_counts().sort_index())

## 5. Understand what each cluster means
Look at the average R, F, M per cluster. Give each a name based on what you see.

In [ ]:
profile = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(1)

print('Average RFM values per cluster:')
print(profile.to_string())
print()
print('How to read this:')
print('  Low Recency  = transacted recently = active customer')
print('  High Recency = not seen in a long time = dormant')
print('  High Frequency + High Monetary = high value customer')

In [ ]:
# Name each cluster based on the profile above
# Cluster 0: Recency=1, Frequency=2160, Monetary=111226 → High Value
# Cluster 1: Recency=276, Frequency=3, Monetary=62     → Dormant
# Cluster 2: Recency=1,   Frequency=489, Monetary=25842 → Low Value
# Cluster 3: Recency=1,   Frequency=1207, Monetary=60038 → Medium Value

cluster_names = {
    0: 'High Value',
    1: 'Dormant',
    2: 'Low Value',
    3: 'Medium Value',
}

rfm['Cluster_Name'] = rfm['Cluster'].map(cluster_names)

print('Named cluster sizes:')
print(rfm['Cluster_Name'].value_counts())

## 6. Visualise the clusters

In [ ]:
# Bar chart — mean R, F, M per cluster
cluster_order = ['High Value', 'Medium Value', 'Low Value', 'Dormant']
colors        = ['#1D9E75', '#2E75B6', '#BA7517', '#888780']

summary = rfm.groupby('Cluster_Name')[['Recency','Frequency','Monetary']].mean().round(1)
summary = summary.reindex(cluster_order)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, label in zip(axes,
    ['Recency','Frequency','Monetary'],
    ['Avg Recency (days since last txn)','Avg Frequency (transactions)','Avg Monetary (total spend $)']):
    ax.bar(cluster_order, summary[col], color=colors, edgecolor='white')
    ax.set_title(label)
    ax.set_ylabel('Value')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('K-Means — What each cluster looks like', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# PCA scatter — see clusters in 2D
pca  = PCA(n_components=2)
X_2d = pca.fit_transform(X)

color_map = {'High Value':'#1D9E75','Medium Value':'#2E75B6',
             'Low Value':'#BA7517','Dormant':'#888780'}

plt.figure(figsize=(11, 7))
for name, color in color_map.items():
    mask = rfm['Cluster_Name'] == name
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=color, label=name, alpha=0.6, s=25)

plt.title('Customer Clusters — 2D view (PCA)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% of variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% of variance)')
plt.legend(title='Cluster', markerscale=2)
plt.tight_layout()
plt.show()

## 7. Evaluate the clustering

> **Important:** K-Means is unsupervised — there is no right or wrong answer to check against. We evaluate using internal metrics.

### Metric 1 — Silhouette Score (most important)
- Ranges −1 to +1
- Above 0.5 = good separation
- Above 0.7 = excellent

In [ ]:
final_sil = silhouette_score(X, rfm['Cluster'])
print(f'Silhouette Score (K=4): {final_sil:.3f}')
print()
if final_sil >= 0.7:
    print('Result: Excellent — clusters are very well separated')
elif final_sil >= 0.5:
    print('Result: Good — clear cluster structure found')
elif final_sil >= 0.3:
    print('Result: Acceptable — some overlap between clusters')
else:
    print('Result: Weak — clusters may not be meaningful')

### Metric 2 — WCSS (Inertia)

In [ ]:
print(f'WCSS (Inertia) for K=4: {km_final.inertia_:.1f}')
print()
print('Lower WCSS = customers are closer to their cluster centre = tighter clusters')
print('Compare this against K=3 and K=5 from the elbow chart above')
print('If K=4 gives a big drop from K=3 and a small drop to K=5, K=4 is the right choice')

### Metric 3 — Cluster size balance

In [ ]:
sizes = rfm['Cluster_Name'].value_counts()
print('Cluster sizes:')
print(sizes)
print()
print('Smallest cluster:', sizes.min(), 'customers')
print('Largest cluster :', sizes.max(), 'customers')
print()
pct_smallest = sizes.min() / len(rfm) * 100
if pct_smallest < 5:
    print('Warning: smallest cluster has less than 5% of customers — may be noise')
else:
    print('Cluster sizes are reasonable — no tiny outlier clusters')

### Metric 4 — Do the clusters make business sense?

In [ ]:
print('=== Final Cluster Summary ===')
print()
final_summary = rfm.groupby('Cluster_Name').agg(
    Customers  = ('Cluster','count'),
    Avg_Recency   = ('Recency','mean'),
    Avg_Frequency = ('Frequency','mean'),
    Avg_Spend     = ('Monetary','mean'),
).round(1)

print(final_summary.sort_values('Avg_Spend', ascending=False).to_string())
print()
print('Ask yourself:')
print('  1. Does High Value have the lowest recency and highest frequency/spend? YES/NO')
print('  2. Does Dormant have the highest recency? YES/NO')
print('  3. Are cluster sizes reasonable (not one tiny group)?')
print('  4. Can you explain what each cluster means in plain English?')
print()
print('If YES to all 4 → K-Means worked well on this data')

In [ ]:
rfm.to_csv('cc_kmeans_output.csv', index=False)
print('Saved: cc_kmeans_output.csv')
print(f'Silhouette Score: {final_sil:.3f}')
print(f'WCSS: {km_final.inertia_:.1f}')